# Spectral scale selection with Rényi divergence

This first experiment:

1. samples $n$ observations from $N(0,1)$;
2. constructs a user-controlled geometric bandwidth grid;
3. evaluates the Gaussian KDE for every bandwidth;
4. computes the Rényi divergence matrix

$$
D_\alpha(p_i \Vert p_j)
= \frac{1}{\alpha-1}
  \log \int p_i(x)^\alpha p_j(x)^{1-\alpha} \, \mathrm{d}x.
$$

5. assembles $S_{ij}=\exp\{-\tau D_\alpha(p_i \Vert p_j)\}$; and
6. plots the stationary distribution of the Doob $h$-transform and its Toeplitz-corrected score.

For $\alpha=1/2$, Rényi divergence is symmetric:

$$
D_{1/2}(p_i \Vert p_j)
= -2 \log \int \sqrt{p_i(x)p_j(x)} \, \mathrm{d}x.
$$

In [26]:
from __future__ import annotations

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from kde_tools.fft import kde_family_fft
from kde_tools.similarities import (
    doob_stationary_mass,
    renyi_divergence_matrix,
    toeplitz_corrected_choice,
    toeplitz_projection,
)

np.set_printoptions(precision=5, suppress=True)

## 1. Experiment parameters

The bandwidth grid is fixed before the sample is generated. Change `h_min`, `h_max`, and `N` to control its range and resolution. The spatial parameters control only the numerical KDE evaluation, not the bandwidth candidates.

In [27]:
seed = 20260818
n = 1000

h_min = 0.01
h_max = 2.00
N = 100

alpha = 0.5
tau = 1.0

half_width = 12.0
spatial_points = 8192

if n < 1:
    raise ValueError("n must be positive.")
if not 0.0 < h_min < h_max:
    raise ValueError("Require 0 < h_min < h_max.")
if N < 2:
    raise ValueError("N must be at least 2.")
if alpha <= 0.0:
    raise ValueError("alpha must be positive.")
if tau <= 0.0:
    raise ValueError("tau must be positive.")

h_grid = np.geomspace(h_min, h_max, N)
rng = np.random.default_rng(seed)
samples = rng.normal(loc=0.0, scale=1.0, size=n)

print(f"sample size: {n}")
print(
    f"bandwidth grid: [{h_grid[0]:.5g}, {h_grid[-1]:.5g}] "
    f"with {len(h_grid)} points"
)

sample size: 1000
bandwidth grid: [0.01, 2] with 100 points


## 2. KDE family, Rényi divergence, and similarity matrix

The KDE family is evaluated on a fine periodic grid by the reusable FFT routine. The endpoint is appended only for trapezoidal integration. At $\alpha=1/2$, $D_\alpha$ should be symmetric; the explicit averaging below removes negligible floating-point asymmetry.

In [28]:
density_values, dx = kde_family_fft(
    samples=samples,
    bandwidths=h_grid,
    half_width=half_width,
    spatial_points=spatial_points,
)

x_grid = np.arange(spatial_points + 1, dtype=float) * dx
periodic_density_values = np.concatenate(
    (density_values, density_values[:, :1]),
    axis=1,
)

D_alpha = renyi_divergence_matrix(
    density_values=periodic_density_values,
    x_grid=x_grid,
    alpha=alpha,
    bandwidths=h_grid,
)

if np.isclose(alpha, 0.5):
    D_alpha = 0.5 * (D_alpha + D_alpha.T)

similarity_matrix = np.exp(-tau * D_alpha)
np.fill_diagonal(similarity_matrix, 1.0)

print(
    "maximum symmetry error in D_alpha: "
    f"{np.max(np.abs(D_alpha - D_alpha.T)):.3e}"
)
print(
    f"similarity range: "
    f"[{similarity_matrix.min():.5f}, {similarity_matrix.max():.5f}]"
)

maximum symmetry error in D_alpha: 0.000e+00
similarity range: [0.67599, 1.00000]


## 3. Doob stationary distribution and Toeplitz correction

If $Sr=\rho r$ and $\ell^\mathsf{T}S=\rho\ell^\mathsf{T}$, the stationary distribution of the Perron–Doob chain is

$$
\pi_i
= \frac{\ell_i r_i}{\sum_j \ell_j r_j}.
$$

For the symmetric $\alpha=1/2$ matrix, $\ell=r$, and hence $\pi_i \propto r_i^2$. The Toeplitz reference averages entries along equal bandwidth-index lags. The existing correction compares the Perron weights through $c_i=r_i/r_i^{(0)}$. Maximizing $c_i$ is equivalent to maximizing the stationary-mass ratio because the latter is proportional to $c_i^2$.

In [29]:
stationary_distribution = doob_stationary_mass(similarity_matrix)
stationary_index = int(np.argmax(stationary_distribution))
stationary_h = float(h_grid[stationary_index])

toeplitz_reference = toeplitz_projection(
    matrix=similarity_matrix,
    symmetric=True,
)
toeplitz_stationary_distribution = doob_stationary_mass(toeplitz_reference)

corrected_h, corrected_index, toeplitz_corrected_score = (
    toeplitz_corrected_choice(
        similarity=similarity_matrix,
        bandwidths=h_grid,
        directed=False,
    )
)

print(
    f"Doob stationary maximum: h = {stationary_h:.6g} "
    f"(index {stationary_index})"
)
print(
    f"Toeplitz-corrected maximum: h = {corrected_h:.6g} "
    f"(index {corrected_index})"
)

Doob stationary maximum: h = 0.401558 (index 69)
Toeplitz-corrected maximum: h = 0.01 (index 0)


In [30]:
fig = make_subplots(
    rows=1,
    cols=3,
    subplot_titles=(
        f"S = exp(-τ D_{alpha:g})",
        "Stationary Doob distribution",
        "Toeplitz-corrected spectral score",
    ),
    horizontal_spacing=0.08,
)

fig.add_trace(
    go.Heatmap(
        z=similarity_matrix,
        colorscale="Viridis",
        colorbar=dict(title="similarity", x=0.29),
        name="similarity",
    ),
    row=1,
    col=1,
)

fig.add_trace(
    go.Scatter(
        x=h_grid,
        y=stationary_distribution,
        mode="lines",
        line=dict(color="#1f77b4", width=3),
        name="Doob stationary distribution π",
    ),
    row=1,
    col=2,
)
fig.add_trace(
    go.Scatter(
        x=h_grid,
        y=toeplitz_stationary_distribution,
        mode="lines",
        line=dict(color="#777777", width=2, dash="dash"),
        name="Toeplitz reference π⁽⁰⁾",
    ),
    row=1,
    col=2,
)
fig.add_trace(
    go.Scatter(
        x=[stationary_h],
        y=[stationary_distribution[stationary_index]],
        mode="markers",
        marker=dict(color="#1f77b4", size=10),
        name=f"stationary maximum: h={stationary_h:.4g}",
    ),
    row=1,
    col=2,
)
fig.add_vline(
    x=stationary_h,
    line=dict(color="#1f77b4", dash="dot"),
    row=1,
    col=2,
)

fig.add_trace(
    go.Scatter(
        x=h_grid,
        y=toeplitz_corrected_score,
        mode="lines",
        line=dict(color="#ff7f0e", width=3),
        name="corrected score r/r⁽⁰⁾",
    ),
    row=1,
    col=3,
)
fig.add_trace(
    go.Scatter(
        x=[corrected_h],
        y=[toeplitz_corrected_score[corrected_index]],
        mode="markers",
        marker=dict(color="#ff7f0e", size=10),
        name=f"corrected maximum: h={corrected_h:.4g}",
    ),
    row=1,
    col=3,
)
fig.add_hline(
    y=1.0,
    line=dict(color="#777777", dash="dash"),
    row=1,
    col=3,
)
fig.add_vline(
    x=corrected_h,
    line=dict(color="#ff7f0e", dash="dot"),
    row=1,
    col=3,
)

fig.update_xaxes(title_text="bandwidth index j", row=1, col=1)
fig.update_yaxes(title_text="bandwidth index i", row=1, col=1)
fig.update_xaxes(title_text="bandwidth h", type="log", row=1, col=2)
fig.update_yaxes(title_text="stationary mass", row=1, col=2)
fig.update_xaxes(title_text="bandwidth h", type="log", row=1, col=3)
fig.update_yaxes(title_text="relative Perron weight", row=1, col=3)

fig.update_layout(
    title=f"N(0,1) sample: n={n}, α={alpha:g}, τ={tau:g}",
    width=1500,
    height=520,
    template="plotly_white",
    legend=dict(orientation="h", y=-0.22),
    margin=dict(t=90, b=130),
)
fig.show()

## 4. Sampling distribution of the selected bandwidth

Repeat the complete selection for $M$ independent samples from $N(0,1)$. Every repetition uses the same fixed geometric bandwidth grid. Record both the uncorrected Doob-stationary maximizer and the Toeplitz-corrected maximizer. Because both selectors return grid points, their grid-point counts give exact empirical distributions.

In [31]:
M = 100
monte_carlo_seed = seed + 1
monte_carlo_rng = np.random.default_rng(monte_carlo_seed)

uncorrected_selected_indices = np.empty(M, dtype=int)
uncorrected_selected_bandwidths = np.empty(M, dtype=float)
selected_indices = np.empty(M, dtype=int)
selected_bandwidths = np.empty(M, dtype=float)

for repetition in range(M):
    repetition_samples = monte_carlo_rng.normal(
        loc=0.0,
        scale=1.0,
        size=n,
    )
    repetition_densities, repetition_dx = kde_family_fft(
        samples=repetition_samples,
        bandwidths=h_grid,
        half_width=half_width,
        spatial_points=spatial_points,
    )
    repetition_x_grid = (
        np.arange(spatial_points + 1, dtype=float) * repetition_dx
    )
    repetition_periodic_densities = np.concatenate(
        (repetition_densities, repetition_densities[:, :1]),
        axis=1,
    )
    repetition_divergence = renyi_divergence_matrix(
        density_values=repetition_periodic_densities,
        x_grid=repetition_x_grid,
        alpha=alpha,
        bandwidths=h_grid,
    )
    if np.isclose(alpha, 0.5):
        repetition_divergence = 0.5 * (
            repetition_divergence + repetition_divergence.T
        )

    repetition_similarity = np.exp(-tau * repetition_divergence)
    np.fill_diagonal(repetition_similarity, 1.0)

    repetition_stationary = doob_stationary_mass(
        repetition_similarity
    )
    uncorrected_index = int(np.argmax(repetition_stationary))
    uncorrected_selected_indices[repetition] = uncorrected_index
    uncorrected_selected_bandwidths[repetition] = h_grid[
        uncorrected_index
    ]

    selected_h, selected_index, _ = toeplitz_corrected_choice(
        similarity=repetition_similarity,
        bandwidths=h_grid,
        directed=False,
    )
    selected_indices[repetition] = selected_index
    selected_bandwidths[repetition] = selected_h

    if (repetition + 1) % 10 == 0 or repetition + 1 == M:
        print(f"completed {repetition + 1}/{M} repetitions")

uncorrected_selection_counts = np.bincount(
    uncorrected_selected_indices, minlength=N
)
selection_counts = np.bincount(selected_indices, minlength=N)
selected_grid_mask = selection_counts > 0
combined_selection_mask = (
    uncorrected_selection_counts + selection_counts
) > 0

print(
    f"uncorrected mean/median: "
    f"{uncorrected_selected_bandwidths.mean():.6g} / "
    f"{np.median(uncorrected_selected_bandwidths):.6g}"
)
print(
    f"uncorrected most frequent bandwidth: "
    f"{h_grid[int(np.argmax(uncorrected_selection_counts))]:.6g}"
)
print(
    f"Toeplitz-corrected mean/median: "
    f"{selected_bandwidths.mean():.6g} / "
    f"{np.median(selected_bandwidths):.6g}"
)
print(
    f"Toeplitz-corrected most frequent bandwidth: "
    f"{h_grid[int(np.argmax(selection_counts))]:.6g}"
)

completed 10/100 repetitions
completed 20/100 repetitions
completed 30/100 repetitions
completed 40/100 repetitions
completed 50/100 repetitions
completed 60/100 repetitions
completed 70/100 repetitions
completed 80/100 repetitions
completed 90/100 repetitions
completed 100/100 repetitions
uncorrected mean/median: 0.395726 / 0.401558
uncorrected most frequent bandwidth: 0.401558
Toeplitz-corrected mean/median: 0.01 / 0.01
Toeplitz-corrected most frequent bandwidth: 0.01


In [32]:
uncorrected_grid_indices = np.flatnonzero(
    uncorrected_selection_counts
)
corrected_grid_indices = np.flatnonzero(selection_counts)
tick_indices = np.unique(
    np.linspace(0, N - 1, 7, dtype=int)
)
tick_bandwidths = [f"{h_grid[index]:.3g}" for index in tick_indices]
maximum_frequency = int(
    max(
        uncorrected_selection_counts.max(),
        selection_counts.max(),
    )
)

histogram = make_subplots(
    rows=1,
    cols=2,
    subplot_titles=(
        "Uncorrected Doob selections",
        "Toeplitz-corrected selections",
    ),
    horizontal_spacing=0.12,
)

histogram.add_trace(
    go.Bar(
        x=uncorrected_grid_indices,
        y=uncorrected_selection_counts[uncorrected_grid_indices],
        customdata=h_grid[uncorrected_grid_indices],
        width=0.85,
        marker=dict(
            color="#1f77b4",
            line=dict(color="white", width=0.5),
        ),
        hovertemplate=(
            "grid index=%{x}"
            "<br>bandwidth h=%{customdata:.6g}"
            "<br>frequency=%{y}<extra></extra>"
        ),
        name="Uncorrected Doob",
    ),
    row=1,
    col=1,
)
histogram.add_trace(
    go.Bar(
        x=corrected_grid_indices,
        y=selection_counts[corrected_grid_indices],
        customdata=h_grid[corrected_grid_indices],
        width=0.85,
        marker=dict(
            color="#ff7f0e",
            line=dict(color="white", width=0.5),
        ),
        hovertemplate=(
            "grid index=%{x}"
            "<br>bandwidth h=%{customdata:.6g}"
            "<br>frequency=%{y}<extra></extra>"
        ),
        name="Toeplitz corrected",
    ),
    row=1,
    col=2,
)

histogram.update_xaxes(
    title_text="selected bandwidth h (geometric grid)",
    range=[-0.5, N - 0.5],
    tickmode="array",
    tickvals=tick_indices,
    ticktext=tick_bandwidths,
    row=1,
    col=1,
)
histogram.update_xaxes(
    title_text="selected bandwidth h (geometric grid)",
    range=[-0.5, N - 0.5],
    tickmode="array",
    tickvals=tick_indices,
    ticktext=tick_bandwidths,
    row=1,
    col=2,
)
histogram.update_yaxes(
    title_text="frequency among M repetitions",
    range=[0, 1.08 * maximum_frequency],
    nticks=8,
    row=1,
    col=1,
)
histogram.update_yaxes(
    title_text="frequency among M repetitions",
    range=[0, 1.08 * maximum_frequency],
    nticks=8,
    row=1,
    col=2,
)
histogram.update_layout(
    title=(
        f"Bandwidth-selection distributions over {M} repetitions "
        f"(n={n}, α={alpha:g}, τ={tau:g})"
    ),
    width=1200,
    height=520,
    template="plotly_white",
    showlegend=False,
)
histogram.show()

## 5. Relative frequency of selecting the lower grid boundary

For every sample size $n$, repeat the Toeplitz-corrected selection `M_per_n` times and estimate

$$
\widehat{p}_{\min}(n)
= \frac{1}{M_{n}}
  \sum_{m=1}^{M_{n}}
  \mathbf{1}\{\widehat{h}_{m,n}=h_{\min}\}.
$$

The comparison uses the selected grid index, so selecting $h_{\min}$ means exactly `selected_index == 0`. The experiment always uses the same fixed bandwidth grid. Its total work is `len(n_values) * M_per_n` complete selections.

In [25]:
n_values = np.array([100, 200, 300, 400, 500], dtype=int)
M_per_n = 10
relative_frequency_seed = seed + 2
relative_frequency_rng = np.random.default_rng(relative_frequency_seed)

h_min_selection_counts = np.zeros(len(n_values), dtype=int)
h_min_relative_frequencies = np.zeros(len(n_values), dtype=float)

for n_position, n_current in enumerate(n_values):
    h_min_count = 0

    for repetition in range(M_per_n):
        current_samples = relative_frequency_rng.normal(
            loc=0.0,
            scale=1.0,
            size=int(n_current),
        )
        current_densities, current_dx = kde_family_fft(
            samples=current_samples,
            bandwidths=h_grid,
            half_width=half_width,
            spatial_points=spatial_points,
        )
        current_x_grid = (
            np.arange(spatial_points + 1, dtype=float) * current_dx
        )
        current_periodic_densities = np.concatenate(
            (current_densities, current_densities[:, :1]),
            axis=1,
        )
        current_divergence = renyi_divergence_matrix(
            density_values=current_periodic_densities,
            x_grid=current_x_grid,
            alpha=alpha,
            bandwidths=h_grid,
        )
        if np.isclose(alpha, 0.5):
            current_divergence = 0.5 * (
                current_divergence + current_divergence.T
            )

        current_similarity = np.exp(-tau * current_divergence)
        np.fill_diagonal(current_similarity, 1.0)
        _, current_selected_index, _ = toeplitz_corrected_choice(
            similarity=current_similarity,
            bandwidths=h_grid,
            directed=False,
        )
        h_min_count += int(current_selected_index == 0)

    h_min_selection_counts[n_position] = h_min_count
    h_min_relative_frequencies[n_position] = h_min_count / M_per_n
    print(
        f"n={n_current}: selected h_min {h_min_count}/{M_per_n} times "
        f"({h_min_relative_frequencies[n_position]:.1%})"
    )

n=100: selected h_min 0/10 times (0.0%)
n=200: selected h_min 1/10 times (10.0%)
n=300: selected h_min 7/10 times (70.0%)
n=400: selected h_min 10/10 times (100.0%)
n=500: selected h_min 10/10 times (100.0%)


In [26]:
h_min_frequency_plot = go.Figure(
    go.Scatter(
        x=n_values,
        y=h_min_relative_frequencies,
        mode="lines+markers",
        line=dict(color="#d62728", width=3),
        marker=dict(color="#d62728", size=10),
        customdata=h_min_selection_counts,
        hovertemplate=(
            "n=%{x}<br>relative frequency=%{y:.1%}"
            "<br>count=%{customdata}/" + str(M_per_n)
            + "<extra></extra>"
        ),
        name="relative frequency of h_min",
    )
)

h_min_frequency_plot.update_xaxes(
    title_text="sample size n",
    tickmode="array",
    tickvals=n_values,
)
h_min_frequency_plot.update_yaxes(
    title_text="relative frequency of selecting h_min",
    tickformat=".0%",
    range=[0.0, 1.05],
)
h_min_frequency_plot.update_layout(
    title=(
        f"Lower-boundary selection frequency "
        f"(M_per_n={M_per_n}, h_min={h_min:g})"
    ),
    width=950,
    height=520,
    template="plotly_white",
    showlegend=False,
)
h_min_frequency_plot.show()

## 6. Shape study for uncorrected Doob selections

This experiment studies how the empirical distribution of the uncorrected Doob-selected scale changes with the Rényi order $\alpha$, sample size $n$, and number $N$ of geometric-grid points. Change the values in the next cell and rerun this section.

For each repetition, the selected index is

$$
\widehat{i}
= \operatorname*{arg\,max}_{0 \leq i < N} \pi_i,
$$

where $\pi$ is the stationary distribution of the uncorrected Perron–Doob chain. No Toeplitz correction is applied. For $\alpha \neq 1/2$, the Rényi divergence matrix is generally directed.

In [33]:
study_alpha = 0.5
study_n = 200
study_N = 100
study_M = 100

study_h_min = 0.01
study_h_max = 2.0
study_tau = 1.0
study_seed = seed + 3
study_half_width = half_width
study_spatial_points = spatial_points

if study_alpha <= 0.0:
    raise ValueError("study_alpha must be positive; use 1.0 for KL.")
if study_n < 1:
    raise ValueError("study_n must be positive.")
if study_N < 2:
    raise ValueError("study_N must be at least 2.")
if study_M < 1:
    raise ValueError("study_M must be positive.")
if not 0.0 < study_h_min < study_h_max:
    raise ValueError("Require 0 < study_h_min < study_h_max.")
if study_tau <= 0.0:
    raise ValueError("study_tau must be positive.")

study_h_grid = np.geomspace(
    study_h_min,
    study_h_max,
    study_N,
)

print(
    f"alpha={study_alpha:g}, n={study_n}, N={study_N}, "
    f"M={study_M}"
)
print(
    f"study bandwidth grid: [{study_h_grid[0]:.5g}, "
    f"{study_h_grid[-1]:.5g}]"
)

alpha=0.5, n=200, N=100, M=100
study bandwidth grid: [0.01, 2]


In [34]:
study_rng = np.random.default_rng(study_seed)
study_selected_indices = np.empty(study_M, dtype=int)
study_selected_bandwidths = np.empty(study_M, dtype=float)

for repetition in range(study_M):
    study_samples = study_rng.normal(
        loc=0.0,
        scale=1.0,
        size=study_n,
    )
    study_densities, study_dx = kde_family_fft(
        samples=study_samples,
        bandwidths=study_h_grid,
        half_width=study_half_width,
        spatial_points=study_spatial_points,
    )
    study_x_grid = (
        np.arange(study_spatial_points + 1, dtype=float)
        * study_dx
    )
    study_periodic_densities = np.concatenate(
        (study_densities, study_densities[:, :1]),
        axis=1,
    )
    study_divergence = renyi_divergence_matrix(
        density_values=study_periodic_densities,
        x_grid=study_x_grid,
        alpha=study_alpha,
        bandwidths=study_h_grid,
    )
    if np.isclose(study_alpha, 0.5):
        study_divergence = 0.5 * (
            study_divergence + study_divergence.T
        )

    study_similarity = np.exp(-study_tau * study_divergence)
    np.fill_diagonal(study_similarity, 1.0)
    study_stationary = doob_stationary_mass(study_similarity)
    study_selected_index = int(np.argmax(study_stationary))
    study_selected_indices[repetition] = study_selected_index
    study_selected_bandwidths[repetition] = study_h_grid[
        study_selected_index
    ]

    if (repetition + 1) % 10 == 0 or repetition + 1 == study_M:
        print(
            f"completed {repetition + 1}/{study_M} repetitions"
        )

study_selection_counts = np.bincount(
    study_selected_indices,
    minlength=study_N,
)
study_selected_grid_indices = np.flatnonzero(study_selection_counts)

print(
    f"mean selected bandwidth: "
    f"{study_selected_bandwidths.mean():.6g}"
)
print(
    f"median selected bandwidth: "
    f"{np.median(study_selected_bandwidths):.6g}"
)
print(
    f"most frequent bandwidth: "
    f"{study_h_grid[int(np.argmax(study_selection_counts))]:.6g}"
)

completed 10/100 repetitions
completed 20/100 repetitions
completed 30/100 repetitions
completed 40/100 repetitions
completed 50/100 repetitions
completed 60/100 repetitions
completed 70/100 repetitions
completed 80/100 repetitions
completed 90/100 repetitions
completed 100/100 repetitions
mean selected bandwidth: 0.143211
median selected bandwidth: 0.127112
most frequent bandwidth: 0.123711


In [35]:
study_tick_indices = np.unique(
    np.linspace(0, study_N - 1, 7, dtype=int)
)
study_tick_bandwidths = [
    f"{study_h_grid[index]:.3g}" for index in study_tick_indices
]

study_histogram = go.Figure(
    go.Bar(
        x=study_selected_grid_indices,
        y=study_selection_counts[study_selected_grid_indices],
        customdata=study_h_grid[study_selected_grid_indices],
        width=0.85,
        marker=dict(
            color="#1f77b4",
            line=dict(color="white", width=0.5),
        ),
        hovertemplate=(
            "grid index=%{x}"
            "<br>bandwidth h=%{customdata:.6g}"
            "<br>frequency=%{y}<extra></extra>"
        ),
        name="uncorrected Doob selections",
    )
)

study_histogram.update_xaxes(
    title_text="selected bandwidth h (geometric grid)",
    range=[-0.5, study_N - 0.5],
    tickmode="array",
    tickvals=study_tick_indices,
    ticktext=study_tick_bandwidths,
)
study_histogram.update_yaxes(
    title_text="frequency among study_M repetitions",
    rangemode="tozero",
    nticks=8,
)
study_histogram.update_layout(
    title=(
        f"Uncorrected Doob bandwidth selections: "
        f"α={study_alpha:g}, n={study_n}, N={study_N}, "
        f"M={study_M}"
    ),
    width=950,
    height=520,
    template="plotly_white",
    showlegend=False,
)
study_histogram.show()